In [1]:
import pandas as pd
import numpy as np
import re


In [2]:
train_path = "train_data.txt"
data = []
with open (train_path, "r", encoding="utf-8") as file:
    for line in file:
        parts = line.split(":::")
        if len(parts) == 4:
            genre = parts[2].strip()
            description = parts[3].strip()
            data.append([genre,description])
train_df = pd.DataFrame(data,columns=["genre","description"])  
train_df.head()          

,genre,description
0,drama,Listening in to a conversation between his doc...
1,thriller,A brother and sister with a past incestuous re...
2,adult,As the bus empties the students for their fiel...
3,drama,To help their unemployed father make ends meet...
4,drama,The film's title refers not only to the un-rec...


In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '',text)
    return text
train_df["clean_desc"] = train_df["description"].apply(clean_text)

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf =TfidfVectorizer(stop_words="english",max_features=5000)
X = tfidf.fit_transform(train_df["clean_desc"])
y = train_df["genre"]

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state=42)

In [6]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train,y_train)

MultinomialNB()

In [7]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
y_pred = model.predict(X_val)
print('Accuracy :', accuracy_score(y_val,y_pred))
print(classification_report(y_val,y_pred))


Accuracy : 0.5209812782440284
              precision    recall  f1-score   support

      action       0.55      0.08      0.14       263
       adult       1.00      0.05      0.10       112
   adventure       0.38      0.04      0.08       139
   animation       0.00      0.00      0.00       104
   biography       0.00      0.00      0.00        61
      comedy       0.51      0.44      0.47      1443
       crime       0.00      0.00      0.00       107
 documentary       0.58      0.88      0.70      2659
       drama       0.45      0.82      0.58      2697
      family       0.00      0.00      0.00       150
     fantasy       0.00      0.00      0.00        74
   game-show       1.00      0.17      0.30        40
     history       0.00      0.00      0.00        45
      horror       0.73      0.36      0.48       431
       music       0.82      0.12      0.22       144
     musical       0.00      0.00      0.00        50
     mystery       0.00      0.00      0.00        

d:\jup\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\jup\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\jup\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [8]:
test_path = "test_data.txt"
test_data = []
with open(test_path,"r",encoding="utf-8") as file:
    for line in file:
        parts = line.split(":::")
        if len(parts) >= 3:
            description = parts[2].strip()
            test_data.append(description)
test_df = pd.DataFrame(test_data,columns=["description"]) 
test_df["clean_desc"] = test_df["description"].apply(clean_text)
         


In [9]:
X_test = tfidf.transform(test_df["clean_desc"])


In [10]:
predictions = model.predict(X_test)
test_df["Predicted Genre"] = predictions
test_df.head()

,description,clean_desc,Predicted Genre
0,"L.R. Brane loves his life - his car, his apart...",lr brane loves his life his car his apartment...,drama
1,"Spain, March 1964: Quico is a very naughty chi...",spain march 1964 quico is a very naughty child...,drama
2,One year in the life of Albin and his family o...,one year in the life of albin and his family o...,documentary
3,"His father has died, he hasn't spoken with his...",his father has died he hasnt spoken with his b...,drama
4,Before he was known internationally as a marti...,before he was known internationally as a marti...,drama


In [11]:
solution = "test_data_solution.txt"
actual_genres = []
with open(solution, "r", encoding="utf-8") as file:
    for line in file:
        parts = line.split(":::")
        if len(parts) >=3:
            genre = parts[2].strip()
            actual_genres.append(genre)
test_df["Actual Genre"] = actual_genres            

In [12]:
accuracy = accuracy_score(test_df["Actual Genre"],test_df["Predicted Genre"])
print ("Test Accuracy :", accuracy)

Test Accuracy : 0.5189852398523985


In [13]:
plot = """A police officer investigating a murder"""
sample_clean = clean_text(plot)
sample_vec = tfidf.transform(["sample_clean"])
prediction = model.predict(sample_vec)
print ("Predicted Genre :", prediction[0])

Predicted Genre : drama
